In [84]:
import json
from dataclasses import dataclass, field
from typing import Optional
import os

from dotenv import load_dotenv
from groq import Groq
from pydantic import BaseModel, Field

In [85]:
load_dotenv("../.env")

client = Groq(
    api_key=os.getenv("GROQ_API_KEY")
)

# from app.services.question_generator import generate_question

In [86]:
class GeneratedQuestion(BaseModel):
    question: str
    topic: str
    difficulty: str
    question_type: str
    expected_concepts: list[str] = Field(
        default_factory=list
    )

class AnswerEvaluation(BaseModel):
    overall_score: float
    technical_accuracy: float
    depth: float
    reasoning: float
    clarity: float
    communication: float
    confidence: float
    strengths: list[str] = Field(default_factory=list)
    weaknesses: list[str] = Field(default_factory=list)
    should_challenge: bool = False
    suggested_follow_up: str = ""
    missing_concepts: list[str] = Field(default_factory=list)

In [87]:
with open(
    "../data/candidate_profile.json",
    "r",
    encoding="utf-8"
) as f:
    candidate = json.load(f)

with open(
    "../data/interview_blueprint.json",
    "r",
    encoding="utf-8"
) as f:
    blueprint = json.load(f)

print("Target role:", blueprint["target_role"])

Target role: Data Scientist


In [88]:
@dataclass
class InterviewState:

    target_role: str

    questions_asked: list[str] = field(
        default_factory=list
    )

    answers: list[str] = field(
        default_factory=list
    )

    evaluations: list[dict] = field(
        default_factory=list
    )
    
    conversation_history: list[dict] = field(
    default_factory=list
)

    topics_covered: list[str] = field(
        default_factory=list
    )

    current_topic: Optional[str] = None

    current_difficulty: str = "medium"

    current_question: Optional[dict] = None

    time_remaining: int = 900

    max_questions: int = 12

    interview_status: str = "not_started"

    follow_up_count: int = 0

    question_count: int = 0

In [89]:
state = InterviewState(
    target_role=blueprint["target_role"]
)

print(state)

InterviewState(target_role='Data Scientist', questions_asked=[], answers=[], evaluations=[], conversation_history=[], topics_covered=[], current_topic=None, current_difficulty='medium', current_question=None, time_remaining=900, max_questions=12, interview_status='not_started', follow_up_count=0, question_count=0)


In [90]:
state.conversation_history.append({
    "role": "interviewer",
    "content": interviewer_response.question
})

state.conversation_history.append({
    "role": "candidate",
    "content": candidate_answer
})

In [91]:
def determine_answer_state(evaluation):

    if evaluation["technical_accuracy"] < 5:
        return "technical_gap"

    if evaluation["depth"] < 5:
        return "shallow"

    if evaluation["reasoning"] < 5:
        return "weak_reasoning"

    if evaluation["overall_score"] >= 8:
        return "strong"

    return "acceptable"

In [92]:
DIFFICULTY_LEVELS = [
    "easy",
    "medium",
    "hard"
]
def adjust_difficulty(
    current_difficulty,
    answer_state
):

    current_index = DIFFICULTY_LEVELS.index(
        current_difficulty
    )

    if answer_state == "strong":

        new_index = min(
            current_index + 1,
            len(DIFFICULTY_LEVELS) - 1
        )

    elif answer_state in [
        "technical_gap",
        "shallow"
    ]:

        new_index = max(
            current_index - 1,
            0
        )

    else:

        new_index = current_index

    return DIFFICULTY_LEVELS[new_index]



In [93]:
def choose_next_topic(
    state,
    blueprint
):

    priority_topics = blueprint[
        "priority_topics"
    ]

    for topic in priority_topics:

        if topic not in state.topics_covered:
            return topic

    # fallback
    for topic in priority_topics:
        return topic

    return "General"

In [94]:
topic = choose_next_topic(
    state,
    blueprint
)

print(topic)

Statistics


In [95]:
def should_follow_up(evaluation):

    if evaluation["should_challenge"]:
        return True

    if evaluation["state"] in [
        "shallow",
        "weak_reasoning"
    ]:
        return True

    return False

In [96]:
MAX_FOLLOW_UPS = 2

def can_follow_up(state):

    return (
        state.follow_up_count
        < MAX_FOLLOW_UPS
    )

In [97]:
def should_end_interview(state):

    if state.question_count >= state.max_questions:
        return True

    if state.time_remaining <= 0:
        return True

    return False

In [98]:
def manager_decision(
    state,
    evaluation=None,
    blueprint=None
):
    
    if should_end_interview(state):

        state.interview_status = "completed"

        return {
            "action": "finish",
            "reason": "Interview limit reached"
        }
    
    # Interview hasn't started
    if state.interview_status == "not_started":

        state.interview_status = "in_progress"

        topic = choose_next_topic(
            state,
            blueprint
        )

        return {
            "action": "ask_question",
            "topic": topic,
            "difficulty": state.current_difficulty,
            "reason": "Start interview"
        }

    # We have an evaluation
    if evaluation is not None:

        answer_state = determine_answer_state(
            evaluation
        )

        # Challenge / follow-up
        if (
            should_follow_up(evaluation)
            and can_follow_up(state)
        ):

            state.follow_up_count += 1

            return {
                "action": "follow_up",
                "topic": state.current_topic,
                "difficulty": state.current_difficulty,
                "reason": answer_state
            }

        # Reset follow-up counter
        state.follow_up_count = 0

        # Adapt difficulty
        state.current_difficulty = (
            adjust_difficulty(
                state.current_difficulty,
                answer_state
            )
        )

        # Choose new topic
        topic = choose_next_topic(
            state,
            blueprint
        )

        return {
            "action": "ask_question",
            "topic": topic,
            "difficulty": state.current_difficulty,
            "reason": answer_state
        }

    return {
        "action": "ask_question",
        "topic": choose_next_topic(
            state,
            blueprint
        ),
        "difficulty": state.current_difficulty,
        "reason": "Default"
    }

In [99]:
decision = manager_decision(
    state,
    blueprint=blueprint
)

print(decision)

{'action': 'ask_question', 'topic': 'Statistics', 'difficulty': 'medium', 'reason': 'Start interview'}


In [100]:
strong_evaluation = {
    "overall_score": 9,
    "technical_accuracy": 9,
    "depth": 9,
    "reasoning": 8,
    "should_challenge": False,
    "state": "strong"
}

decision = manager_decision(
    state,
    evaluation=strong_evaluation,
    blueprint=blueprint
)

print(decision)

{'action': 'ask_question', 'topic': 'Statistics', 'difficulty': 'hard', 'reason': 'strong'}


In [101]:
weak_evaluation = {
    "overall_score": 4,
    "technical_accuracy": 4,
    "depth": 3,
    "reasoning": 4,
    "should_challenge": False,
    "state": "technical_gap"
}

decision = manager_decision(
    state,
    evaluation=weak_evaluation,
    blueprint=blueprint
)

print(decision)

{'action': 'ask_question', 'topic': 'Statistics', 'difficulty': 'medium', 'reason': 'technical_gap'}


In [102]:
shallow_evaluation = {
    "overall_score": 6,
    "technical_accuracy": 7,
    "depth": 4,
    "reasoning": 4,
    "should_challenge": False,
    "state": "shallow"
}

decision = manager_decision(
    state,
    evaluation=shallow_evaluation,
    blueprint=blueprint
)

print(decision)

{'action': 'follow_up', 'topic': None, 'difficulty': 'medium', 'reason': 'shallow'}


In [103]:
suspicious_evaluation = {
    "overall_score": 7,
    "technical_accuracy": 7,
    "depth": 6,
    "reasoning": 6,
    "should_challenge": True,
    "state": "acceptable"
}

decision = manager_decision(
    state,
    evaluation=suspicious_evaluation,
    blueprint=blueprint
)

print(decision)

{'action': 'follow_up', 'topic': None, 'difficulty': 'medium', 'reason': 'acceptable'}


In [104]:
def should_end_interview(state):

    if state.question_count >= state.max_questions:
        return True

    if state.time_remaining <= 0:
        return True

    return False

In [105]:
state.question_count = 12

print(
    should_end_interview(state)
)

True


In [106]:
state = InterviewState(
    target_role=blueprint["target_role"]
)

decision = manager_decision(
    state,
    blueprint=blueprint
)
print(decision)

{'action': 'ask_question', 'topic': 'Statistics', 'difficulty': 'medium', 'reason': 'Start interview'}


In [107]:
def generate_question(
    topic: str,
    difficulty: str,
    question_type: str,
    previous_questions: list[str] | None = None
):
    
    if previous_questions is None:
        previous_questions = []

    previous_text = "\n".join(
        f"- {q}"
        for q in previous_questions[-10:]
    )

    prompt = f"""
You are an expert technical interviewer.

Generate ONE interview question.

Target topic:
{topic}

Difficulty:
{difficulty}

Question type:
{question_type}

Previously asked questions:
{previous_text if previous_text else "None"}

Rules:

1. The question must test the specified topic.
2. Match the requested difficulty.
3. Do not repeat or closely rephrase previous questions.
4. The question should be appropriate for a technical interview.
5. Return expected concepts that a strong answer should contain.
6. Return ONLY valid JSON.

Return:

{{
    "question": "...",
    "topic": "...",
    "difficulty": "...",
    "question_type": "...",
    "expected_concepts": [
        "...",
        "..."
    ]
}}
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an expert technical interviewer. "
                    "Return only valid JSON."
                )
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.7
    )

    response_text = response.choices[0].message.content

    result = json.loads(response_text)

    return GeneratedQuestion.model_validate(result)

In [108]:
question = generate_question(
    topic=decision["topic"],
    difficulty=decision["difficulty"],
    question_type="technical",
    previous_questions=state.questions_asked
)

In [109]:
print(question.question)
print(question.expected_concepts)

You have two independent samples drawn from normal populations with unknown means and unknown variances. How would you test whether the two population means are equal? Describe the steps, assumptions, the appropriate test statistic, how you would handle the case of unequal variances, and how you would interpret the result.
['Independent two‑sample t‑test', 'Assumptions: independence, approximate normality of each sample', 'Pooled variance estimator for equal variances', "Welch's t‑test for unequal variances (different variance estimate and degrees of freedom)", 'Formulas for the test statistic and degrees of freedom', 'Decision rule using p‑value or critical value', 'Interpretation of the test outcome and confidence interval for the difference of means']


In [110]:
candidate_answer = """
I chose XGBoost because it is a gradient boosting algorithm
that works very well with structured and tabular data. It can
capture nonlinear relationships and interactions between
features. It also provides useful feature importance and
usually performs well without requiring extensive feature
scaling.
"""

In [111]:
question_for_evaluation = question.model_dump()
print(
    json.dumps(
        question_for_evaluation,
        indent=2
    )
)

{
  "question": "You have two independent samples drawn from normal populations with unknown means and unknown variances. How would you test whether the two population means are equal? Describe the steps, assumptions, the appropriate test statistic, how you would handle the case of unequal variances, and how you would interpret the result.",
  "topic": "Statistics",
  "difficulty": "medium",
  "question_type": "technical",
  "expected_concepts": [
    "Independent two\u2011sample t\u2011test",
    "Assumptions: independence, approximate normality of each sample",
    "Pooled variance estimator for equal variances",
    "Welch's t\u2011test for unequal variances (different variance estimate and degrees of freedom)",
    "Formulas for the test statistic and degrees of freedom",
    "Decision rule using p\u2011value or critical value",
    "Interpretation of the test outcome and confidence interval for the difference of means"
  ]
}


In [112]:
evaluation_schema = AnswerEvaluation.model_json_schema()

print(json.dumps(
    evaluation_schema,
    indent=2
))

{
  "properties": {
    "overall_score": {
      "title": "Overall Score",
      "type": "number"
    },
    "technical_accuracy": {
      "title": "Technical Accuracy",
      "type": "number"
    },
    "depth": {
      "title": "Depth",
      "type": "number"
    },
    "reasoning": {
      "title": "Reasoning",
      "type": "number"
    },
    "clarity": {
      "title": "Clarity",
      "type": "number"
    },
    "communication": {
      "title": "Communication",
      "type": "number"
    },
    "confidence": {
      "title": "Confidence",
      "type": "number"
    },
    "strengths": {
      "items": {
        "type": "string"
      },
      "title": "Strengths",
      "type": "array"
    },
    "weaknesses": {
      "items": {
        "type": "string"
      },
      "title": "Weaknesses",
      "type": "array"
    },
    "should_challenge": {
      "default": false,
      "title": "Should Challenge",
      "type": "boolean"
    },
    "suggested_follow_up": {
      "default":

In [113]:
EVALUATOR_PROMPT = """
You are the Answer Evaluation Agent for InterviewHive.

You are an expert technical interviewer evaluating a candidate's
answer to an interview question.

Evaluate ONLY the candidate's answer against:
- the interview question
- the expected concepts
- the target role when relevant

Do not evaluate the candidate as a person.

EVALUATION DIMENSIONS

Score every dimension from 0 to 10.

1. technical_accuracy
   - Are the technical statements correct?
   - Penalize incorrect technical claims strongly.

2. depth
   - How thoroughly does the answer explain the concept?
   - A short but correct answer can still score well.

3. reasoning
   - Does the candidate explain why, how, trade-offs,
     implications, or decision-making?

4. clarity
   - Is the answer understandable and logically organized?

5. communication
   - Does the candidate communicate the answer effectively,
     directly, and professionally?

6. confidence
   - Evaluate how appropriately and decisively the answer
     is communicated.
   - Do not invent confidence evidence.

OVERALL SCORE

overall_score must represent the overall quality of the answer.

Technical accuracy and relevance to the question are especially
important.

SCORING GUIDE

0-2 = very poor
3-4 = weak
5-6 = acceptable
7-8 = strong
9-10 = excellent

STRENGTHS

Only mention things the candidate actually did well.

WEAKNESSES

Mention specific problems in the answer.

MISSING CONCEPTS

Only include concepts from expected_concepts that are genuinely
missing or insufficiently addressed.

Do not invent missing concepts.

CHALLENGE

Set should_challenge to true if:
- the candidate makes an incorrect technical claim,
- the answer is too vague to establish understanding,
- an important concept is misunderstood,
- or a follow-up would meaningfully test the candidate.

Otherwise set it to false.

FOLLOW-UP

If should_challenge is true, provide a useful follow-up question
targeting the biggest weakness.

If no follow-up is needed, return an empty string.

IMPORTANT OUTPUT RULES

Return ONLY valid JSON.

Use EXACTLY these field names:

{
    "overall_score": 0,
    "technical_accuracy": 0,
    "depth": 0,
    "reasoning": 0,
    "clarity": 0,
    "communication": 0,
    "confidence": 0,
    "strengths": [],
    "weaknesses": [],
    "should_challenge": false,
    "suggested_follow_up": "",
    "missing_concepts": []
}

Do NOT use alternative field names such as:
- score
- feedback
- accuracy
- follow_up
- explanation

Every field must be present.

Return ONLY the JSON object.
"""

In [114]:
def evaluate_answer(
    question,
    candidate_answer
):

    evaluation_input = {
        "target_role": blueprint["target_role"],

        "question": question["question"],

        "topic": question.get("topic", ""),

        "category": question.get("category", ""),

        "difficulty": question.get("difficulty", ""),

        "question_type": question.get("question_type", ""),

        "expected_concepts": question.get(
            "expected_concepts",
            []
        ),

        "candidate_answer": candidate_answer
    }

    prompt = f"""
Evaluate the candidate's interview answer using the provided
question and expected concepts.

QUESTION CONTEXT
{json.dumps(
    evaluation_input,
    indent=2,
    ensure_ascii=False
)}

REQUIRED JSON SCHEMA
{json.dumps(
    evaluation_schema,
    indent=2
)}

Return ONLY valid JSON matching the schema.
"""

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",

        messages=[
            {
                "role": "system",
                "content": EVALUATOR_PROMPT
            },
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0
    )

    response_text = response.choices[0].message.content

    result = json.loads(response_text)

    return AnswerEvaluation.model_validate(result)

In [115]:
evaluation = evaluate_answer(
    question_for_evaluation,
    candidate_answer
)

print(evaluation.model_dump())

{'overall_score': 1.0, 'technical_accuracy': 0.0, 'depth': 0.0, 'reasoning': 0.0, 'clarity': 2.0, 'communication': 2.0, 'confidence': 2.0, 'strengths': [], 'weaknesses': ['The answer does not address the statistical hypothesis testing question at all.', 'It discusses XGBoost, which is unrelated to testing equality of two population means.', 'No mention of independent two‑sample t‑test, assumptions, test statistic, handling unequal variances, or interpretation.'], 'should_challenge': True, 'suggested_follow_up': 'Can you describe the appropriate statistical test for comparing the means of two independent normal samples with unknown and possibly unequal variances, including assumptions, test statistic formulas, and how you would interpret the result?', 'missing_concepts': ['Independent two‑sample t‑test', 'Assumptions: independence, approximate normality of each sample', 'Pooled variance estimator for equal variances', "Welch's t‑test for unequal variances (different variance estimate an

In [116]:
def determine_answer_state(evaluation):

    if evaluation["technical_accuracy"] < 5:
        return "technical_gap"

    if evaluation["depth"] < 5:
        return "shallow"

    if evaluation["reasoning"] < 5:
        return "weak_reasoning"

    if evaluation["overall_score"] >= 8:
        return "strong"

    return "acceptable"

In [117]:
# WEIGHTS = {
#     "technical_accuracy": 0.30,
#     "depth": 0.20,
#     "reasoning": 0.20,
#     "clarity": 0.10,
#     "communication": 0.10,
#     "confidence": 0.10
# }
# def calculate_weighted_score(evaluation):

#     score = (
#         evaluation.technical_accuracy
#         * WEIGHTS["technical_accuracy"]

#         + evaluation.depth
#         * WEIGHTS["depth"]

#         + evaluation.reasoning
#         * WEIGHTS["reasoning"]

#         + evaluation.clarity
#         * WEIGHTS["clarity"]

#         + evaluation.communication
#         * WEIGHTS["communication"]

#         + evaluation.confidence
#         * WEIGHTS["confidence"]
#     )

#     return round(score, 2)


In [118]:
def create_evaluation_signal(evaluation):

    answer_state = determine_answer_state(
        evaluation.model_dump()
    )

    return {
        "overall_score": evaluation.overall_score,
        "technical_accuracy": evaluation.technical_accuracy,
        "depth": evaluation.depth,
        "reasoning": evaluation.reasoning,
        "clarity": evaluation.clarity,
        "communication": evaluation.communication,
        "confidence": evaluation.confidence,
        "state": answer_state,
        "should_challenge": evaluation.should_challenge,
        "suggested_follow_up": evaluation.suggested_follow_up,
        "missing_concepts": evaluation.missing_concepts
    }

In [119]:
signal = create_evaluation_signal(
    evaluation
)

print(
    json.dumps(
        signal,
        indent=2,
        ensure_ascii=False
    )
)

{
  "overall_score": 1.0,
  "technical_accuracy": 0.0,
  "depth": 0.0,
  "reasoning": 0.0,
  "clarity": 2.0,
  "communication": 2.0,
  "confidence": 2.0,
  "state": "technical_gap",
  "should_challenge": true,
  "suggested_follow_up": "Can you describe the appropriate statistical test for comparing the means of two independent normal samples with unknown and possibly unequal variances, including assumptions, test statistic formulas, and how you would interpret the result?",
  "missing_concepts": [
    "Independent two‑sample t‑test",
    "Assumptions: independence, approximate normality of each sample",
    "Pooled variance estimator for equal variances",
    "Welch's t‑test for unequal variances (different variance estimate and degrees of freedom)",
    "Formulas for the test statistic and degrees of freedom",
    "Decision rule using p‑value or critical value",
    "Interpretation of the test outcome and confidence interval for the difference of means"
  ]
}


In [120]:
state.answers.append(
    candidate_answer
)

state.evaluations.append(
    signal
)

In [121]:
print("Answers:", len(state.answers))
print("Evaluations:", len(state.evaluations))

Answers: 1
Evaluations: 1


In [122]:
next_decision = manager_decision(
    state,
    evaluation=signal,
    blueprint=blueprint
)

print(
    json.dumps(
        next_decision,
        indent=2
    )
)

{
  "action": "follow_up",
  "topic": null,
  "difficulty": "medium",
  "reason": "technical_gap"
}


In [123]:
next_question = generate_question(
    topic=next_decision["topic"],
    difficulty=next_decision["difficulty"],
    question_type="technical",
    previous_questions=state.questions_asked
)
print(next_question.question)

Given an array of integers (which may include both positive and negative numbers), write a function to find the length of the longest contiguous subarray whose elements sum to zero. Explain your approach and analyze its time and space complexity.


In [124]:
def check_question_history(
    new_question: str,
    asked_questions: list[str],
    threshold: float = 0.80
):

    if not asked_questions:
        return {
            "is_duplicate": False,
            "similarity": 0.0,
            "matched_question": None
        }

    new_embedding = model.encode(
        [new_question],
        normalize_embeddings=True
    ).astype("float32")

    history_embeddings = model.encode(
        asked_questions,
        normalize_embeddings=True
    ).astype("float32")

    scores = np.matmul(
        history_embeddings,
        new_embedding[0]
    )

    best_index = int(
        np.argmax(scores)
    )

    best_score = float(
        scores[best_index]
    )

    return {
        "is_duplicate": best_score >= threshold,
        "similarity": best_score,
        "matched_question": asked_questions[best_index]
    }

In [125]:
def is_question_repetitive(
    new_question,
    previous_questions,
    threshold=0.85
):
    
    if not previous_questions:
        return False

    result = check_question_history(
        new_question,
        previous_questions,
        threshold=threshold
    )

    return result

In [126]:
def generate_unique_question(
    topic,
    difficulty,
    question_type,
    previous_questions,
    max_attempts=3
):

    for attempt in range(max_attempts):

        question = generate_question(
            topic=topic,
            difficulty=difficulty,
            question_type=question_type,
            previous_questions=previous_questions
        )

        is_duplicate = is_question_repetitive(
            question.question,
            previous_questions
        )

        if not is_duplicate:
            return question

        print(
            f"Question too similar. "
            f"Regenerating ({attempt + 1}/{max_attempts})..."
        )

    # If all attempts fail, return the last generated question
    return question

In [127]:
question = generate_question(
    topic=decision["topic"],
    difficulty=decision["difficulty"],
    question_type="technical",
    previous_questions=state.questions_asked
)

In [128]:
print(question.question)

Suppose you have a random sample X₁, X₂, ..., Xₙ drawn i.i.d. from a Normal(μ, σ²) distribution where the variance σ² is known. Derive the maximum likelihood estimator (MLE) for the mean μ, prove that this estimator is unbiased, and argue why it achieves the minimum variance among all unbiased estimators of μ.


In [129]:
state.current_question = question.model_dump()

state.questions_asked.append(
    question.question
)

state.current_topic = question.topic

if question.topic not in state.topics_covered:
    state.topics_covered.append(question.topic)

state.question_count += 1

In [130]:
decision = manager_decision(
    state,
    blueprint=blueprint
)

print(decision)

{'action': 'ask_question', 'topic': 'Probability', 'difficulty': 'medium', 'reason': 'Default'}


In [131]:
question = generate_question(
    topic=decision["topic"],
    difficulty=decision["difficulty"],
    question_type="technical",
    previous_questions=state.questions_asked
)

print(question.question)

Let X and Y be independent exponential random variables with rate λ (i.e., X, Y ~ Exp(λ)). Define Z = min(X, Y) and I = 1_{\{X < Y\}} (the indicator that X is the smaller of the two). 

(a) Derive the joint probability mass/density function of (Z, I). 
(b) Compute the expected value E[Z]. 
(c) Explain how the memoryless property of the exponential distribution simplifies the calculation in part (b).


In [132]:
candidate_answer = """
I would evaluate the model using appropriate classification
metrics such as precision, recall and F1 score. The choice
depends on whether false positives or false negatives are
more important for the application.
"""

In [133]:
question_for_evaluation = question.model_dump()

evaluation = evaluate_answer(
    question_for_evaluation,
    candidate_answer
)

signal = create_evaluation_signal(
    evaluation
)

In [134]:
state.answers.append(candidate_answer)
state.evaluations.append(signal)

In [135]:
next_decision = manager_decision(
    state,
    evaluation=signal,
    blueprint=blueprint
)

print(next_decision)

{'action': 'follow_up', 'topic': 'Statistics', 'difficulty': 'medium', 'reason': 'technical_gap'}


In [136]:
def run_interview_turn(
    state,
    blueprint,
    candidate_answer=None
):

    #Evaluate previous answer

    evaluation = None
    signal = None

    if candidate_answer is not None:

        current_question = state.current_question

        evaluation = evaluate_answer(
            current_question,
            candidate_answer
        )

        signal = create_evaluation_signal(
            evaluation
        )

        state.answers.append(
            candidate_answer
        )

        state.evaluations.append(
            signal
        )

    # Manager decides next action

    decision = manager_decision(
        state,
        evaluation=signal,
        blueprint=blueprint
    )

    # Check termination

    if decision["action"] == "finish":

        state.interview_status = "completed"

        return {
            "status": "completed",
            "decision": decision,
            "question": None,
            "evaluation": (
                evaluation.model_dump()
                if evaluation
                else None
            )
        }

    #Generate next question

    question = generate_question(
        topic=decision["topic"],
        difficulty=decision["difficulty"],
        question_type="technical",
        previous_questions=state.questions_asked
    )

    # Update state

    state.current_question = (
        question.model_dump()
    )

    state.current_topic = question.topic

    state.current_difficulty = (
        question.difficulty
    )

    state.questions_asked.append(
        question.question
    )

    if question.topic not in state.topics_covered:

        state.topics_covered.append(
            question.topic
        )

    state.question_count += 1

    #Return turn

    return {
        "status": "in_progress",

        "decision": decision,

        "question": question.model_dump(),

        "evaluation": (
            evaluation.model_dump()
            if evaluation
            else None
        )
    }

In [137]:
state = InterviewState(
    target_role=blueprint["target_role"]
)

turn = run_interview_turn(
    state,
    blueprint
)

print(turn["question"]["question"])

You have a random sample of 200 observations from an unknown population. The sample mean is 50 and the sample standard deviation is 10. You want to test, at the 5% significance level, whether the population mean μ is greater than 45. Describe the steps you would take, compute the appropriate test statistic, determine the critical value or p‑value, and state your conclusion.


In [138]:
turn = run_interview_turn(
    state,
    blueprint,
    candidate_answer="""
    I would use cross validation because it gives a better
    estimate of how the model will generalize to unseen data.
    """
)

print(turn["decision"])
print(turn["question"]["question"])

{'action': 'follow_up', 'topic': 'Statistics', 'difficulty': 'medium', 'reason': 'technical_gap'}
A researcher collects data from two independent groups. Group A has n₁ = 45 observations with a sample mean of 78 and a sample standard deviation of 12. Group B has n₂ = 60 observations with a sample mean of 85 and a sample standard deviation of 15. Assuming the two populations are approximately normal but may have unequal variances, test at the 1% significance level whether the population mean of Group B is greater than that of Group A. Describe the steps you would follow, compute the appropriate test statistic, determine the critical value or p‑value using Welch’s t‑test, and state your conclusion.


In [139]:
turn = run_interview_turn(
    state,
    blueprint,
    candidate_answer="""
    I think cross validation is good because it tests the model.
    """
)

print(turn["evaluation"])
print(turn["decision"])
print(turn["question"]["question"])

{'overall_score': 1.0, 'technical_accuracy': 0.0, 'depth': 0.0, 'reasoning': 0.0, 'clarity': 2.0, 'communication': 2.0, 'confidence': 2.0, 'strengths': ['Clear and concise phrasing'], 'weaknesses': ["Answer does not address the Welch's t‑test problem presented in the question", 'Provides an unrelated concept (cross‑validation) instead of statistical hypothesis testing', 'Fails to formulate hypotheses, compute test statistic, degrees of freedom, or draw a conclusion'], 'should_challenge': True, 'suggested_follow_up': "Please walk through the steps of performing Welch's t‑test for the two groups, including hypothesis formulation, calculation of the test statistic, degrees of freedom using the Welch–Satterthwaite equation, and interpretation of the p‑value at the 1% significance level.", 'missing_concepts': ['Formulating null and alternative hypotheses for a two‑sample mean comparison', 'Choosing Welch’s t‑test due to unequal variances', 'Calculating the test statistic', 'Computing the ap

In [140]:
INTERVIEWER_PROMPT = """
You are the interviewer in a realistic technical job interview.

Your job is to communicate naturally with the candidate.

You receive:
- candidate profile
- target role
- manager decision
- generated question
- previous conversation

Rules:

1. Ask exactly one question at a time.
2. Be professional but conversational.
3. Do not give the candidate the answer.
4. Do not evaluate the candidate.
5. Do not reveal internal scores or agent decisions.
6. If the action is a follow-up, connect naturally to the candidate's
   previous answer.
7. If the action is a new question, transition naturally.
8. Questions should sound like something a real interviewer would ask.
9. Reference the candidate's resume when relevant.
10. Keep the response concise.

Return ONLY valid JSON:

{
    "message": "...",
    "question": "..."
}
"""

In [141]:
class InterviewerResponse(BaseModel):

    message: str
    question: str

In [142]:
def interviewer_agent(
    decision,
    question,
    candidate_profile,
    conversation_history=None,
    candidate_answer=None
):

    if conversation_history is None:
        conversation_history = []

    input_data = {
        "candidate_profile": candidate_profile,
        "manager_decision": decision,
        "question": question,
        "conversation_history": conversation_history[-6:],
        "candidate_answer": candidate_answer
    }

    response = client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[
            {
                "role": "system",
                "content": INTERVIEWER_PROMPT
            },
            {
                "role": "user",
                "content": json.dumps(
                    input_data,
                    indent=2,
                    ensure_ascii=False
                )
            }
        ],
        temperature=0.6
    )

    response_text = response.choices[0].message.content

    parsed = json.loads(response_text)

    return InterviewerResponse.model_validate(
        parsed
    )

In [143]:
with open(
    "../data/candidate_profile.json",
    "r",
    encoding="utf-8"
) as f:
    candidate_profile = json.load(f)

print(candidate_profile)

{'name': 'Bhumi Saraogi', 'email': 'saraogibhumi@gmail.com', 'phone': '(+91) 9531657378', 'education': [{'degree': 'B.Tech. in Computer Science and Engineering', 'institution': 'NSHM Knowledge Campus, Durgapur', 'cgpa': 7.28, 'graduation_year': None}], 'skills': ['Python', 'Java', 'JavaScript', 'SQL', 'Pandas', 'NumPy', 'Matplotlib', 'Seaborn', 'Scikit-learn', 'PyTorch', 'Deep Learning', 'NLP', 'Computer Vision', 'Transformers', 'LLMs', 'Prompt Engineering', 'Embeddings', 'RAG', 'React.js', 'HTML', 'CSS', 'Tailwind CSS', 'Node.js', 'Express.js', 'FastAPI', 'REST APIs', 'MongoDB', 'Supabase', 'Git', 'GitHub', 'Jupyter Notebook', 'Streamlit'], 'projects': [{'name': 'SceneSense AI', 'description': 'Engineered a semantic search engine using Sentence Transformers to generate dense vector embeddings for natural language movie retrieval. Implemented FAISS-based vector indexing and cosine similarity search, enabling low-latency, context-aware recommendations beyond keyword matching. Developed 

In [144]:
interviewer_response = interviewer_agent(
    decision=turn["decision"],
    question=turn["question"],
    candidate_profile=candidate_profile,
    conversation_history=state.answers
)

In [145]:
print(
    interviewer_response.message
)

print(
    interviewer_response.question
)

Thanks for sharing your background in data analysis and machine learning, Bhumi. Let's dive into the statistics question.
Can you describe how you would formulate the null and alternative hypotheses for the chi‑square goodness‑of‑fit test to assess whether this die is fair?
